In [1]:
# ================================================================
#  LCCN-R15 - LANGUAGE-IDENTIFICATION AUDIT AND ROUTER SENSITIVITY
#  Answers IJIES Reviewer 1, point 5.  [CPU, ~2.5 h]
#  The router is pipeline.is_english(w) = zipf_frequency(w,'en') >= 2.5;
#  tokens below the threshold are tagged romanized_kannada and routed to
#  KOPE / phonetic normalization.
#  PHASE 1 (runs now):
#    Cell 3  stratified audit sheet: 200 Latin-script training TYPES per
#            Zipf band (<1.5, 1.5-2.5, 2.5-3.5, 3.5-4.5, >=4.5) with two
#            contexts each -> langid_audit_sheet.csv for two annotators.
#    Cell 4  router sensitivity: thresholds 2.0 / 2.5 / 3.0 / 3.5 and
#            (an alternative identifier, IndicLID, is not installable as
#            a package, so the sweep is over the threshold);
#            cascade rebuilt each time; unified accounting; LR and SVM.
#  PHASE 2 (after annotation): attach the completed sheet as a dataset
#    (file name langid_audit_done.csv, columns ann1 and ann2 filled with
#    E = English, K = romanized Kannada, O = other/name/unclear) and
#    re-run: Cell 5 scores precision, recall, F1 of the romanized-Kannada
#    tag at every threshold, Cohen's kappa, and per-band error rates.
#  Add Input: paper1-nb1-lccn-final (output), paper1-nb2-lccn-ml
#             (output, trainer.py), final-emotion-dataset, gold-ds,
#             and in Phase 2 the dataset holding langid_audit_done.csv.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 3.3 MB/s eta 0:00:00
python 3.13.15


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
assert cand, "attach the paper1-nb1-lccn-final output"
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP (replicates NB1 CELL 2; frozen split asserted)
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | hash {TEST_HASH}")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | hash e2d6e1f7d564cace


In [4]:
# ================================================================
#  CELL 3 - PHASE 1: STRATIFIED AUDIT SHEET
# ================================================================
import re, random
from collections import Counter, defaultdict
from wordfreq import zipf_frequency
random.seed(42)

train = df[df[ID].isin(SPLIT["train"])]
types = Counter(); ctx = defaultdict(list)
for cid, raw in zip(train[ID], train["text_raw"]):
    toks = raw.split()
    for i, t in enumerate(toks):
        if t.isalpha() and t.isascii() and len(t) >= 2:
            w = t.lower(); types[w] += 1
            if len(ctx[w]) < 6:
                ctx[w].append(" ".join(toks[max(0, i-4):i+5]))
print(f"Latin-script training types: {len(types):,} | token occurrences {sum(types.values()):,}")

BANDS = [("<1.5", 0.0, 1.5), ("1.5-2.5", 1.5, 2.5), ("2.5-3.5", 2.5, 3.5), ("3.5-4.5", 3.5, 4.5), (">=4.5", 4.5, 99)]
zipf = {w: zipf_frequency(w, "en") for w in types}
by_band = defaultdict(list)
for w, z in zipf.items():
    for name, lo, hi in BANDS:
        if lo <= z < hi: by_band[name].append(w); break
tok_by_band = {b: sum(types[w] for w in ws) for b, ws in by_band.items()}
print("\nband      types     tokens   tag at 2.5")
for name, lo, hi in BANDS:
    print(f"{name:<9}{len(by_band[name]):>7,}{tok_by_band[name]:>11,}   {'english' if lo >= 2.5 else 'romanized_kannada'}")

rows = []
for name, lo, hi in BANDS:
    pool = by_band[name]; k = min(200, len(pool))
    for w in random.sample(pool, k):
        cs = random.sample(ctx[w], min(2, len(ctx[w])))
        rows.append(dict(token=w, zipf=round(zipf[w], 2), band=name, train_freq=types[w],
                         tag_at_2_5="english" if zipf[w] >= 2.5 else "romanized_kannada",
                         context1=cs[0] if cs else "", context2=cs[1] if len(cs) > 1 else "",
                         ann1="", ann2="", note=""))
sheet = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
sheet.to_csv("/kaggle/working/langid_audit_sheet.csv", index=False, encoding="utf-8-sig")
print(f"\nwritten langid_audit_sheet.csv: {len(sheet)} rows (shuffled; band column retained for scoring)")
print("Annotator instruction: ann1/ann2 = E (English word, incl. slang/abbreviation), "
      "K (romanized Kannada, incl. Kannada name/word in Latin script), O (other: non-Kannada name, code, unclear).")


Latin-script training types: 26,484 | token occurrences 196,813

band      types     tokens   tag at 2.5
<1.5      18,511     49,129   romanized_kannada
1.5-2.5    1,808     16,896   romanized_kannada
2.5-3.5    1,594     14,057   english
3.5-4.5    2,486     24,976   english
>=4.5      2,085     91,755   english

written langid_audit_sheet.csv: 1000 rows (shuffled; band column retained for scoring)
Annotator instruction: ann1/ann2 = E (English word, incl. slang/abbreviation), K (romanized Kannada, incl. Kannada name/word in Latin script), O (other: non-Kannada name, code, unclear).


In [5]:
# ================================================================
#  CELL 4 - PHASE 1: ROUTER SENSITIVITY (threshold sweep + IndicLID)
#  Monkeypatches pipeline.is_english, rebuilds the full cascade at the
#  operating point (theta 3, delta 2, alpha .01), unified accounting.
# ================================================================
from tqdm.auto import tqdm
single = {f: next(iter(e)) for f, e in F2E.items() if len(e) == 1}
train_mask = df[ID].isin(SPLIT["train"]).values
df_tr, df_rest = df[train_mask], df[~train_mask]
_IS_EN = pipeline.is_english

def route_share(frame):
    n = k = 0
    for t in frame[TXT]:
        toks, tags = process_comment(t)
        for tok, tg in zip(toks, tags):
            if tok.isalpha() and tok.isascii(): n += 1; k += (tg == "romanized_kannada")
    return k, n

def accounting(name, o):
    # raw tokenization under the ACTIVE router: segment_latin also calls
    # is_english, so the token stream can differ from the stored text_raw
    raw_tr   = [" ".join(process_comment(t)[0]) for t in df_tr[TXT]]
    raw_rest = [" ".join(process_comment(t)[0]) for t in df_rest[TXT]]
    tr_out = [o.normalize(t) for t in tqdm(df_tr[TXT], desc=name, leave=False)]
    forms_rewritten = len({k[1] for k, v in o.cache.items() if v != k[1]})
    blocked = {k: v for k, v in o.stats.items() if k.startswith("blocked")}
    rest_out = [o.normalize(t) for t in df_rest[TXT]]
    ch = tot = 0
    for a, b in zip(raw_tr + raw_rest, tr_out + rest_out):
        ra, rb = a.lower().split(), b.lower().split(); assert len(ra) == len(rb), name
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    mp = defaultdict(Counter)
    for a, b in zip(raw_tr, tr_out):
        for w, x in zip(a.split(), b.split()): mp[w.lower()][x.lower()] += 1
    buckets = defaultdict(set)
    for f, c in mp.items(): buckets[c.most_common(1)[0][0]].add(f)
    pairs = lab = same = 0
    for forms in buckets.values():
        n = len(forms)
        if n < 2: continue
        pairs += n*(n-1)//2
        ce = Counter(single[f] for f in forms if f in single); L = sum(ce.values())
        lab += L*(L-1)//2; same += sum(k*(k-1)//2 for k in ce.values())
    x = lab - same; prec = 1 - x/lab if lab else float("nan")
    full = pd.Series(tr_out + rest_out, index=list(df_tr.index) + list(df_rest.index)).reindex(df.index)
    seg_diff = sum(1 for a, b in zip(raw_tr + raw_rest, list(df_tr["text_raw"]) + list(df_rest["text_raw"])) if a != b)
    return dict(tokens_changed_full_pct=round(100*ch/tot, 2), comments_resegmented=seg_diff, forms_rewritten_train=forms_rewritten,
                merged_pairs_train=pairs, x_cross_train=x, merge_precision_train=round(prec, 4), **blocked), full

indiclid = None   # IndicLID is not distributed as an installable package; the sweep covers the sensitivity question

ROUTERS = [("zipf_2.0", lambda w: zipf_frequency(w.lower(), "en") >= 2.0),
           ("zipf_2.5_operating", _IS_EN),
           ("zipf_3.0", lambda w: zipf_frequency(w.lower(), "en") >= 3.0),
           ("zipf_3.5", lambda w: zipf_frequency(w.lower(), "en") >= 3.5)]

rows = []
for name, fn in ROUTERS:
    pipeline.is_english = fn
    try:
        k, n = route_share(df)
        o = LCCN(norm_map, evidence, make_veto(F2E, LEX), translit=None, lex=LEX)
        row, texts = accounting(name, o)
    finally:
        pipeline.is_english = _IS_EN
    df[f"text_router_{name}"] = texts.values
    row = dict(router=name, latin_tokens_routed_to_phonetic_pct=round(100*k/n, 2), **row)
    rows.append(row)
    print(f"{name:<20} routed {row['latin_tokens_routed_to_phonetic_pct']:5.2f}%  resegmented {row['comments_resegmented']:>5,}  tok {row['tokens_changed_full_pct']:5.2f}%  "
          f"forms {row['forms_rewritten_train']:>6,}  pairs {row['merged_pairs_train']:>6,}  X {row['x_cross_train']}  prec {row['merge_precision_train']}")
sens = pd.DataFrame(rows); sens.to_csv("/kaggle/working/router_sensitivity.csv", index=False)

# classical macro-F1 per router
import shutil, glob, os, importlib
tc = [p for p in sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)) if "nb2" in p]
tc = tc or sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True)); assert tc
shutil.copy(tc[0], "/kaggle/working/trainer.py"); sys.modules.pop("trainer", None)
import trainer; importlib.reload(trainer); from trainer import *
assert trainer.TRAINER_VERSION == "1.4"
sd = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
tr, va, te, HAVE = load_splits(sd[0])
NEW = [f"text_router_{n}" for n, _ in ROUTERS]
lut = {c: dict(zip(df[ID].astype(str), df[c])) for c in NEW}
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for c in NEW: f[c] = ids.map(lut[c]); assert f[c].notna().all(), c
ARMS = ["raw", "lccn"] + [f"router_{n}" for n, _ in ROUTERS]
res = run_grid(tr, va, te, ARMS, ["Logistic Regression", "SVM"], tag="lccn_r15_router", resume=False)
r_lr, r_svm = res.iloc[0], res.iloc[1]
print("\n=== TABLE 12: router sensitivity ===")
print(f"{'router':<22}{'routed%':>9}{'tok%':>7}{'forms':>7}{'pairs':>7}{'X':>4}{'LR':>8}{'SVM':>8}{'mean':>8}")
for (n, _), row in zip(ROUTERS, rows):
    a = f"router_{n}"; m = (float(r_lr[a]) + float(r_svm[a])) / 2
    print(f"{n:<22}{row['latin_tokens_routed_to_phonetic_pct']:>9.2f}{row['tokens_changed_full_pct']:>7.2f}"
          f"{row['forms_rewritten_train']:>7,}{row['merged_pairs_train']:>7,}{row['x_cross_train']:>4}"
          f"{float(r_lr[a]):>8.4f}{float(r_svm[a]):>8.4f}{m:>8.4f}")
print(f"{'lccn (NB1 texts)':<22}{'':>9}{'':>7}{'':>7}{'':>7}{'':>4}{float(r_lr['lccn']):>8.4f}{float(r_svm['lccn']):>8.4f}")
print("router_zipf_2.5_operating must match lccn (0.8017 / 0.8022) up to the one order-dependent decision.")


zipf_2.0:   0%|          | 0/28000 [00:00<?, ?it/s]

zipf_2.0             routed 29.22%  resegmented     1  tok 14.12%  forms  4,645  pairs  2,031  X 1  prec 0.9988


zipf_2.5_operating:   0%|          | 0/28000 [00:00<?, ?it/s]

zipf_2.5_operating   routed 32.57%  resegmented     0  tok 14.16%  forms  4,680  pairs  2,050  X 1  prec 0.9988


zipf_3.0:   0%|          | 0/28000 [00:00<?, ?it/s]

zipf_3.0             routed 35.88%  resegmented     4  tok 14.22%  forms  4,710  pairs  2,056  X 1  prec 0.9988


zipf_3.5:   0%|          | 0/28000 [00:00<?, ?it/s]

zipf_3.5             routed 39.42%  resegmented     7  tok 14.29%  forms  4,741  pairs  2,062  X 1  prec 0.9988
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

=== Logistic Regression / raw ===
  -> macro-F1 0.8033

=== Logistic Regression / lccn ===
  -> macro-F1 0.8017

=== Logistic Regression / router_zipf_2.0 ===
  -> macro-F1 0.8009

=== Logistic Regression / router_zipf_2.5_operating ===
  -> macro-F1 0.8013

=== Logistic Regression / router_zipf_3.0 ===
  -> macro-F1 0.8015


In [6]:
# ================================================================
#  CELL 5 - PHASE 2: SCORE THE COMPLETED AUDIT (runs only if attached)
# ================================================================
done = glob.glob("/kaggle/input/**/langid_audit_done.csv", recursive=True)
if not done:
    print("langid_audit_done.csv not attached: Phase 2 skipped. Attach the completed sheet and re-run.")
else:
    a = pd.read_csv(done[0], dtype=str, keep_default_na=False)
    a.columns = [c.strip().lower() for c in a.columns]
    for c in ("ann1", "ann2"): a[c] = a[c].str.strip().str.upper().str[:1]
    valid = a[a.ann1.isin(list("EKO")) & a.ann2.isin(list("EKO"))].copy()
    print(f"rows {len(a)} | both annotated {len(valid)}")
    # Cohen's kappa
    cats = ["E", "K", "O"]; n = len(valid)
    po = (valid.ann1 == valid.ann2).mean()
    pe = sum((valid.ann1 == c).mean() * (valid.ann2 == c).mean() for c in cats)
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    print(f"Cohen's kappa (E/K/O): {kappa:.3f}  (observed agreement {po:.3f})")
    gold = valid[valid.ann1 == valid.ann2].copy()
    gold["is_k"] = gold.ann1 == "K"; gold["zipf"] = gold.zipf.astype(float); gold["w"] = gold.train_freq.astype(float)
    print(f"adjudicated gold (agreement only): {len(gold)} tokens; disagreements {len(valid) - len(gold)} listed in langid_disagreements.csv")
    valid[valid.ann1 != valid.ann2].to_csv("/kaggle/working/langid_disagreements.csv", index=False)

    print("\nromanized-Kannada tag vs gold, by threshold (type-level and token-weighted):")
    print(f"{'thr':>5}{'P':>8}{'R':>8}{'F1':>8}{'P_w':>8}{'R_w':>8}{'F1_w':>8}")
    for thr in (2.0, 2.5, 3.0, 3.5):
        pred = gold.zipf < thr
        def prf(wt):
            tp = (wt * (pred & gold.is_k)).sum(); fp = (wt * (pred & ~gold.is_k)).sum(); fn = (wt * (~pred & gold.is_k)).sum()
            p = tp/(tp+fp) if tp+fp else 0; r = tp/(tp+fn) if tp+fn else 0; f = 2*p*r/(p+r) if p+r else 0
            return p, r, f
        p, r, f = prf(np.ones(len(gold))); pw, rw, fw = prf(gold.w.values)
        print(f"{thr:>5.1f}{p:>8.3f}{r:>8.3f}{f:>8.3f}{pw:>8.3f}{rw:>8.3f}{fw:>8.3f}")

    print("\nerror composition by band at the 2.5 operating threshold:")
    gold["pred_k"] = gold.zipf < 2.5
    for b in ["<1.5", "1.5-2.5", "2.5-3.5", "3.5-4.5", ">=4.5"]:
        g = gold[gold.band == b]
        if not len(g): continue
        en_routed = int((g.pred_k & ~g.is_k).sum()); kn_missed = int((~g.pred_k & g.is_k).sum())
        print(f"  {b:<9} n {len(g):>4}  English routed to phonetic {en_routed:>3} ({100*en_routed/len(g):4.1f}%)  "
              f"romanized Kannada tagged English {kn_missed:>3} ({100*kn_missed/len(g):4.1f}%)")
    gold.to_csv("/kaggle/working/langid_gold.csv", index=False)
    print("\nDONE - langid_gold.csv, langid_disagreements.csv written; transcribe only from the committed log.")


rows 1000 | both annotated 1000
Cohen's kappa (E/K/O): 0.906  (observed agreement 0.954)
adjudicated gold (agreement only): 954 tokens; disagreements 46 listed in langid_disagreements.csv

romanized-Kannada tag vs gold, by threshold (type-level and token-weighted):
  thr       P       R      F1     P_w     R_w    F1_w
  2.0   0.801   0.708   0.752   0.965   0.615   0.751
  2.5   0.757   0.854   0.803   0.950   0.698   0.805
  3.0   0.685   0.958   0.799   0.932   0.902   0.917
  3.5   0.593   0.997   0.744   0.869   0.998   0.929

error composition by band at the 2.5 operating threshold:
  <1.5      n  192  English routed to phonetic  21 (10.9%)  romanized Kannada tagged English   0 ( 0.0%)
  1.5-2.5   n  187  English routed to phonetic  71 (38.0%)  romanized Kannada tagged English   0 ( 0.0%)
  2.5-3.5   n  186  English routed to phonetic   0 ( 0.0%)  romanized Kannada tagged English  48 (25.8%)
  3.5-4.5   n  189  English routed to phonetic   0 ( 0.0%)  romanized Kannada tagged Engli